
## GET Started With Faiss VECTOR STORE

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    DirectoryLoader
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

True

### DATA INGESTION AND PROCESSING

In [2]:
from langchain_core.documents import Document

sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is a branch of computer science that focuses on building systems
        capable of performing tasks that normally require human intelligence. These tasks include reasoning,
        learning, problem-solving, perception, and decision-making. AI is used in chatbots, recommendation systems,
        autonomous vehicles, healthcare diagnostics, and many other applications.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),

    Document(
        page_content="""
        Machine Learning (ML) is a subset of Artificial Intelligence that enables computers to learn
        patterns from data without being explicitly programmed. Common types of machine learning include
        supervised learning, unsupervised learning, and reinforcement learning. ML is widely used in spam filtering,
        fraud detection, price prediction, and recommendation systems.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),

    Document(
        page_content="""
        Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers
        to understand, interpret, and generate human language. NLP techniques are used in chatbots,
        language translation, sentiment analysis, text summarization, and question-answering systems.
        Modern NLP models are often based on transformer architectures such as BERT and GPT.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    ),

    Document(
        page_content="""
        Deep Learning (DL) is a specialized area of Machine Learning that uses neural networks with
        multiple hidden layers to learn complex patterns from large amounts of data. Deep Learning has
        achieved significant success in image recognition, speech recognition, medical imaging, and
        autonomous driving. Popular frameworks include TensorFlow and PyTorch.
        """,
        metadata={"source": "DL Basics", "page": 1, "topic": "DL"}
    )
]

print(f"Total Documents: {len(sample_documents)}")
print(f"MetaData:{sample_documents[0].metadata}")
print(f"PageContent:{sample_documents[0].page_content[:120]}")

Total Documents: 4
MetaData:{'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
PageContent:
        Artificial Intelligence (AI) is a branch of computer science that focuses on building systems
        capable o


### TEXT SPLITTING

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "],
)

chunks = text_splitter.split_documents(sample_documents)
chunks[0]

Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is a branch of computer science that focuses on building systems\n        capable of performing tasks that normally require human intelligence. These tasks include reasoning,\n        learning, problem-solving, perception, and decision-making. AI is used in chatbots, recommendation systems,\n        autonomous vehicles, healthcare diagnostics, and many other applications.')

### Initialize Embedding Model

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-miniLM-L6-V2"
)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Create FAISS VectoreStore

In [5]:
vectorestore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
vectorestore

In [6]:
vectorestore.save_local('faiss_index')
print("Vectorestore Save into 'FAISS_index' Directory")

Vectorestore Save into 'FAISS_index' Directory


In [7]:
loaded_vectorestor  = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True    
)

In [8]:
## Similarity Search
query = "What is deep learning "
results = vectorestore.similarity_search(query,k=3)
print(results)

[Document(id='6757219c-00e6-4bae-9763-f6096085e84e', metadata={'source': 'DL Basics', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning (DL) is a specialized area of Machine Learning that uses neural networks with\n        multiple hidden layers to learn complex patterns from large amounts of data. Deep Learning has\n        achieved significant success in image recognition, speech recognition, medical imaging, and\n        autonomous driving. Popular frameworks include TensorFlow and PyTorch.'), Document(id='4d26ee5d-29b7-423a-9380-f9c78a3a595b', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning (ML) is a subset of Artificial Intelligence that enables computers to learn\n        patterns from data without being explicitly programmed. Common types of machine learning include\n        supervised learning, unsupervised learning, and reinforcement learning. ML is widely used in spam filtering,\n        fraud detection, price prediction, and re

### Building Rag Chain With LECL

In [9]:
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm  = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002703C84C050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002703C84D6A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
simple_prompt = ChatPromptTemplate.from_template(
    """ 
    Answer The Following question Based only on the following 
    context:{context}
    Question:{question}
  Answer:  """)

In [13]:
retriever= vectorestore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":3}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002703D94D2B0>, search_kwargs={'k': 3})

In [14]:
from typing import List
def format_docs(docs:List[Document]) -> str:
    """Format Document For Instruction into prompt """
    
    formatted = []
    for i,doc in enumerate(docs):
        source = doc.metadata.get('Source',"unknown")
        formatted.append(f"Document {i+1} (Source:{source}):\n{doc.page_content}")
        
    
    return "\n\n .join(formatted)"    
        

In [15]:
simple_rag_chain = (
    {"context":retriever | format_docs , "question":RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [16]:
conversational_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful AI assistant.use the provided context to answer the question."),
        ("placeholder","{chat_history}"),
        ("human","Context: {context}\n\n Question : {input}"),
    ]
)

In [17]:
def create_conversational_rag():
    return(
        RunnablePassthrough.assign(
            context = lambda x : format_docs(retriever.invoke(x["input"]))
        )
        |conversational_prompt
        | llm
        | StrOutputParser()
    )
    
conversational_rag = create_conversational_rag()

In [26]:
## conversational rag
chat_history = []
q1 = "ehat do you mean by data structure"
a1 = conversational_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(a1)

A **data structure** is a way of organizing, storing, and managing data in a computer so that it can be accessed and manipulated efficiently.  

In programming, a data structure defines both:

1. **The layout of the data** – how individual elements are arranged (e.g., in a list, tree, graph, table, etc.).
2. **The operations that can be performed** – what you can do with the data (e.g., insert, delete, search, traverse, sort, etc.) and how efficiently those operations run.

### Why It Matters
- **Performance:** The choice of data structure directly affects the speed (time complexity) and memory usage (space complexity) of algorithms.
- **Clarity:** Using the right structure makes code easier to understand and maintain.
- **Functionality:** Certain problems naturally fit specific structures (e.g., a priority queue for task scheduling, a graph for network routing).

### Common Examples

| Category | Example | Typical Use |
|----------|---------|-------------|
| **Linear** | **Array / Lis

In [27]:
from langchain.messages import AIMessage , HumanMessage
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [28]:
## conversational rag
chat_history = []
q1 = "who invented data structure"
a1 = conversational_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(a1)

The short answer is that **no single person “invented” data structures**—they grew out of many early ideas in mathematics, engineering, and computer science and were refined by a succession of researchers over several decades.  

Below is a quick timeline of the most influential milestones and the people who introduced the concepts that later became the core of modern data‑structure theory:

| Era | Key Contribution | Who Introduced It | Why It Matters |
|-----|------------------|-------------------|----------------|
| **1800s – early computing concepts** | The notion of **arrays** (ordered collections of items) and **records** (grouped fields) | **Charles Babbage** (Analytical Engine) & **Ada Lovelace** (programming ideas) | First formal description of storing and accessing data in a systematic way. |
| **1940s – early electronic computers** | **Stack** (LIFO) and **Queue** (FIFO) concepts appear in hardware design | **John von Neumann** (EDVAC) and later **Floyd H. Kahn** (stack impl